# TrustOCT: Trustworthy Cross-Scanner Retinal OCT Diagnostics Framework
### Master Orchestrator Notebook (Clean Model Naming v1.1)
---
### 📌 TABLE OF CONTENTS
*   [SECTION 1: Environment Setup](#section1)
*   [SECTION 2: Dataset Verification & Statistics](#section2)
*   [SECTION 3: Preprocessing & Denoising](#section3)
*   [SECTION 4: Baseline Models Setup](#section4)
*   [SECTION 5: Proposed TrustOCT Model](#section5)
*   [SECTION 6: Model Training Execution](#section6)
*   [SECTION 7: Classification Evaluation](#section7)
*   [SECTION 8: Ablation Study](#section8)
*   [SECTION 9: LayerCAM Visual Attributions](#section9)
*   [SECTION 10: Faithfulness Evaluation (Deletion & Insertion)](#section10)
*   [SECTION 11: External Validation (OCTID)](#section11)
*   [SECTION 12: Statistical Analysis & Wilcoxon Significance](#section12)
*   [SECTION 13: Publication Paper Assets](#section13)
---
<a id='section1'></a>
## SECTION 1: Environment Setup


In [ ]:
import os

# 1. Clone repository if missing (for Google Colab runners)
if not os.path.exists('src'):
    print('🔄 Cloning Trustworthy-OCT-AI repository...')
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    print('Running within the active repository workspace.')

# 2. Install dependencies
try:
    import albumentations
    import ptflops
    print('✅ All dependency packages are already loaded.')
except ImportError:
    print('🔄 Installing required libraries...')
    !pip install -r requirements.txt


In [ ]:
# Setup imports
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')


In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running locally. Skipping Google Drive mount.')


<a id='section2'></a>
## SECTION 2: Dataset Verification & Statistics
Loads the training mappings for Kermany OCT2017 and dynamically computes **Table 1: Dataset Statistics**.


In [ ]:
# Dataset Download via Kaggle API (for Google Colab runners)
if not os.path.exists('/content/Kermany') and not os.path.exists('Kermany'):
    try:
        from google.colab import files
        print("🔑 Please upload your kaggle.json API token file:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print("🔄 Downloading Kermany OCT2017 dataset from Kaggle...")
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print("✅ Kermany OCT2017 successfully downloaded and extracted to /content/Kermany.")
        else:
            print("❌ Upload failed. kaggle.json was not found.")
    except Exception as e:
        print("Running locally or download skipped. Ensure Kermany folder is present.")


In [ ]:
# Load dataset mappings dynamically (Self-Healing Cell)
from src.datasets.dataset import auto_detect_columns, patient_level_split

drive_base = '/content/drive/MyDrive'
csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join(drive_base, 'kermany_dataset_mapping.csv')

if not os.path.exists(csv_path):
    print('🔄 CSV mapping file missing. Generating dynamically from directories...')
    root_oct = '/content/Kermany/OCT2017 ' # Kaggle extract path (with trailing space)
    if not os.path.exists(root_oct):
        root_oct = '/content/Kermany'
    if not os.path.exists(root_oct):
        root_oct = 'e:/aiml master/OCT2017'
        csv_path = 'e:/aiml master/kermany_dataset_mapping.csv'
    
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files in os.walk(root_oct):
            for f in files:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({
                            'image_path': os.path.join(root, f),
                            'label': lbl,
                            'patient_id': pt_id
                        })
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)
        print(f'✅ Success: Dynamically created mapping CSV with {len(df_new)} images.')

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    sample_img_path = df.iloc[0]['image_path']
    if not os.path.exists(sample_img_path):
        print('Attempting automatic path correction...')
        def convert_path_to_colab(win_path):
            linux_path = win_path.replace('\\', '/')
            # Correct path matching Kaggle's extraction subfolder:
            relative_path = linux_path[linux_path.find('OCT2017 /'):] if 'OCT2017 /' in linux_path else '/'.join(linux_path.split('/')[-3:])
            return os.path.join('/content/Kermany', relative_path)
        df['image_path'] = df['image_path'].apply(convert_path_to_colab)
        sample_img_path = df.iloc[0]['image_path']
    
    train_df, val_df, test_df = patient_level_split(df)
    print(f'Dataset successfully loaded. Train shape: {train_df.shape}')
    print(f'Does sample image exist? {os.path.exists(sample_img_path)}')
else:
    # Local fallback for code validation
    print('Dataset files not found. Initializing mock dataset for notebook compilation checks...')
    mock_records = []
    for idx in range(100):
        mock_records.append({
            'image_path': f'dummy_{idx}.jpg',
            'label': idx % 4,
            'patient_id': f'Pat_{idx % 15}'
        })
    df = pd.DataFrame(mock_records)
    train_df, val_df, test_df = patient_level_split(df)


In [ ]:
total = len(df)
print(f"Train: {len(train_df)} ({100*len(train_df)/total:.2f}%)")
print(f"Validation: {len(val_df)} ({100*len(val_df)/total:.2f}%)")
print(f"Test: {len(test_df)} ({100*len(test_df)/total:.2f}%)")


In [ ]:
print("Unique patients")
print("Train:", train_df["patient_id"].nunique())
print("Validation:", val_df["patient_id"].nunique())
print("Test:", test_df["patient_id"].nunique())


In [ ]:
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))


In [ ]:
# Patient Leakage Check
train_patients = set(train_df['patient_id'].unique())
test_patients = set(test_df['patient_id'].unique())
leakage = train_patients.intersection(test_patients)
print(f'Patient overlap count: {len(leakage)}')
assert len(leakage) == 0, 'CRITICAL error: Patient data leakage detected!'
print('Success: Zero patient leakage verified.')


In [ ]:
# Compute Table 1 dynamically (mapping indices back to class names)
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
table_1 = df.groupby('label').agg(
    total_images=('image_path', 'count'),
    unique_patients=('patient_id', 'nunique')
).reset_index().rename(columns={'label': 'Diagnostic Class', 'total_images': 'Total Images', 'unique_patients': 'Unique Patients'})
table_1['Diagnostic Class'] = table_1['Diagnostic Class'].apply(lambda x: CLASSES[int(x)])
print('--- TABLE 1: DATASET STATISTICS (COMPUTED) ---')
display(table_1)
os.makedirs('results/tables', exist_ok=True)
table_1.to_csv('results/tables/table_1_dataset_statistics.csv', index=False)


<a id='section3'></a>
## SECTION 3: Preprocessing & Denoising
Applies and visualizes edge-preserving speckle denoising (Bilateral filtering) on training B-scans.


In [ ]:
# Visualizing Preprocessing
from src.preprocessing.filters import bilateral_filter

sample_row = df.iloc[0]
sample_path = sample_row['image_path']
if os.path.exists(sample_path):
    raw_img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
    processed_img = bilateral_filter(raw_img)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(raw_img, cmap='gray'); axes[0].set_title('Original B-scan')
    axes[0].axis('off')
    axes[1].imshow(processed_img, cmap='gray'); axes[1].set_title('Bilateral Denoised')
    axes[1].axis('off')
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/figure_1_preprocessing.png', dpi=300)
    plt.show()
else:
    print('Local image file missing. Skipping preprocessing plot.')


<a id='section4'></a>
## SECTION 4: Baseline Models Setup
Loads the comparison backbones (ResNet-50, DenseNet-121, etc.).


In [ ]:
from src.models.builder import TrustOCTModel

# Configure pre-trained architectures from modular builder
resnet_baseline = TrustOCTModel(
    backbone_name='resnet50', feature_module='identity', attention_module='identity',
    dg_module='identity', head_name='softmax', num_classes=4, pretrained=True
).to(device)
print('Pre-trained baselines loaded successfully.')


<a id='section5'></a>
## SECTION 5: Proposed TrustOCT Model
Loads the attention-gated ResNet backbone with sequentially integrated Channel-Spatial Attention (CBAM) and MultiScale Fusion.


In [ ]:
trust_oct_model = TrustOCTModel(
    backbone_name='resnet50', feature_module='multiscale', attention_module='cbam',
    dg_module='mixstyle', head_name='evidential', num_classes=4, pretrained=True
).to(device)
print('TrustOCT model successfully initialized with pre-trained weights.')


<a id='section6'></a>
## SECTION 6: Model Training Execution
Trains each baseline model and the proposed TrustOCT model separately under identical hyperparameters.


In [ ]:
#@title Training Configuration Form
experiment_id = 'trustoct' #@param ['resnet50', 'msf_resnet50', 'msf_cbam_resnet50', 'msf_cbam_mixstyle_resnet50', 'msf_cbam_edl_resnet50', 'trustoct']
train_all_experiments = True #@param {type: 'boolean'}
epochs = 30 #@param {type:'integer'}
lr = 1e-4 #@param {type:'number'}
batch_size = 32 #@param {type:'integer'}

from src.training.trainer import run_experiment

# Train the selected experiments (automatically loops if train_all_experiments is True)
experiments = ['resnet50', 'msf_resnet50', 'msf_cbam_resnet50', 'msf_cbam_mixstyle_resnet50', 'msf_cbam_edl_resnet50', 'trustoct'] if train_all_experiments else [experiment_id]

for exp_id in experiments:
    run_experiment(
        experiment_id=exp_id,
        train_df=train_df,
        val_df=val_df,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        device_name=str(device)
    )


<a id='section7'></a>
## SECTION 7: Classification Evaluation
Evaluates **all models** dynamically by loading their best checkpoints, and compiles the comparison **Table 2** with 95% Bootstrap Confidence Intervals. Also exports individual confusion matrices and reliability diagrams.


In [ ]:
from src.evaluation.classification import evaluate_classification_metrics
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.models.builder import build_model
from src.preprocessing.standardizer import RetinalPipelineTransform
from src.datasets.dataset import RetinalDataset
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram

val_transform = RetinalPipelineTransform(is_training=False)
test_dataset = RetinalDataset(test_df, transform=val_transform, apply_bilateral=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

def compute_all_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    
    perf = evaluate_classification_metrics(labels, preds)
    confidences = np.max(probs, axis=1)
    accuracies = (preds == labels).astype(int)
    
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    
    # Specificity macro
    from src.evaluation.classification import compute_multiclass_specificity
    specificity = compute_multiclass_specificity(labels, preds)
    
    # ROC-AUC ovr macro
    present_classes = sorted(list(np.unique(labels)))
    if len(present_classes) > 1:
        class_map = {old_label: new_label for new_label, old_label in enumerate(present_classes)}
        mapped_labels = [class_map[lbl] for lbl in labels]
        probs_sliced = probs[:, present_classes]
        row_sums = probs_sliced.sum(axis=1, keepdims=True)
        probs_sliced = np.where(row_sums > 1e-5, probs_sliced / row_sums, np.ones_like(probs_sliced) / probs_sliced.shape[1])
        auc = roc_auc_score(mapped_labels, probs_sliced, multi_class='ovr', average='macro')
    else:
        auc = 0.5
        
    return {
        'Accuracy': perf['accuracy'], 'Precision': perf['precision'], 'Recall': perf['recall'],
        'Specificity': specificity, 'Macro F1': perf['f1_score'], 'Kappa': perf['cohens_kappa'],
        'ROC-AUC': auc, 'ECE': ece, 'Brier': brier
    }

def evaluate_and_bootstrap(config, weight_path, n_bootstraps=200):
    model_inst = build_model(config).to(device)
    is_ev = (config['model']['head'] == 'evidential')
    if os.path.exists(weight_path):
        model_inst.load_state_dict(torch.load(weight_path, map_location=device))
    model_inst.eval()
    
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model_inst(images)
            if is_ev:
                from src.models.edl_head import get_evidence_metrics
                p, _ = get_evidence_metrics(outputs)
            else:
                p = torch.softmax(outputs, dim=1)
            
            probs.extend(p.cpu().numpy())
            _, predicted = torch.max(p, 1)
            preds.extend(predicted.cpu().numpy())
            labels.extend(targets.numpy())
            
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    
    base_scores = compute_all_metrics(labels, preds, probs)
    bootstrap_results = {k: [] for k in base_scores.keys()}
    n_samples_len = len(labels)
    np.random.seed(42)
    for _ in range(n_bootstraps):
        boot_idx = np.random.choice(n_samples_len, size=n_samples_len, replace=True)
        boot_labels = labels[boot_idx]
        boot_preds = preds[boot_idx]
        boot_probs = probs[boot_idx]
        
        boot_scores = compute_all_metrics(boot_labels, boot_preds, boot_probs)
        for k, v in boot_scores.items():
            bootstrap_results[k].append(v)
            
    report = {}
    for k, base_val in base_scores.items():
        boot_vals = sorted(bootstrap_results[k])
        ci_lower = boot_vals[int(0.025 * n_bootstraps)]
        ci_upper = boot_vals[int(0.975 * n_bootstraps)]
        report[k] = base_val
        report[f'{k}_CI'] = (ci_lower, ci_upper)
        
    # Save individual figures (confusion matrix & reliability curve) in model directory
    exp_dir = os.path.dirname(weight_path)
    if os.path.exists(exp_dir):
        cm_path = os.path.join(exp_dir, 'confusion_matrix.png')
        plot_confusion_matrix(labels, preds, CLASSES, cm_path)
        rd_path = os.path.join(exp_dir, 'reliability_diagram.png')
        plot_reliability_diagram(probs, labels, rd_path)
        
    return report, preds, labels, probs

models_to_evaluate = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'TrustOCT (Proposed)')
]

comparison_rows = []
best_preds, best_labels, best_probs = None, None, None

for config_item, path, display_name in models_to_evaluate:
    print(f'Auditing {display_name} with bootstrap CIs...')
    report, preds, labels, probs = evaluate_and_bootstrap(config_item, path)
    
    row = {'Model': display_name}
    for metric in ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Macro F1', 'Kappa', 'ROC-AUC', 'ECE', 'Brier']:
        val = report[metric]
        ci = report[f"{metric}_CI"]
        row[metric] = f"{val:.4f} ({ci[0]:.4f} - {ci[1]:.4f})"
    comparison_rows.append(row)
    
    if display_name == 'TrustOCT (Proposed)':
        best_preds = preds
        best_labels = labels
        best_probs = probs
        
table_2_df = pd.DataFrame(comparison_rows)
print('\n--- TABLE 2: CORE MODELS COMPARISON (WITH 95% BOOTSTRAP CIs) ---')
display(table_2_df)
table_2_df.to_csv('results/tables/table_2_metrics_ci.csv', index=False)


In [ ]:
# Display Confusion Matrix for Proposed Model
from PIL import Image
cm_path = 'outputs/trustoct/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image.open(cm_path))


<a id='section8'></a>
## SECTION 8: Ablation Study
Evaluates ablation weight checkpoints dynamically to compile **Table 3: Ablation Study**.


In [ ]:
from src.evaluation.classification import evaluate_classification_metrics
from sklearn.metrics import matthews_corrcoef

def evaluate_checkpoint(config, weight_path, loader):
    if not os.path.exists(weight_path):
        return "-", "-", "-"
    model_inst = build_model(config).to(device)
    model_inst.load_state_dict(torch.load(weight_path, map_location=device))
    model_inst.eval()
    is_ev = (config['model']['head'] == 'evidential')
    
    preds, labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            outputs = model_inst(images)
            if is_ev:
                from src.models.edl_head import get_evidence_metrics
                probs, _ = get_evidence_metrics(outputs)
            else:
                probs = torch.softmax(outputs, dim=1)
            _, p = torch.max(probs, 1)
            preds.extend(p.cpu().numpy())
            labels.extend(targets.numpy())
            
    perf = evaluate_classification_metrics(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    return f"{perf['accuracy']*100:.2f}%", f"{perf['f1_score']:.4f}", f"{mcc:.4f}"

ablation_configs = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'trustoct')
]

ablation_rows = []
for config_item, path, config_name in ablation_configs:
    acc_val, f1_val, mcc_val = evaluate_checkpoint(config_item, path, test_loader)
    ablation_rows.append({
        'Configuration': config_name,
        'Accuracy (%)': acc_val,
        'Macro F1': f1_val,
        'MCC': mcc_val
    })
    
ablation_df = pd.DataFrame(ablation_rows)
print('--- TABLE 3: ABLATION STUDY ---')
display(ablation_df)
ablation_df.to_csv('results/tables/table_3_ablation_study.csv', index=False)


<a id='section9'></a>
## SECTION 9: LayerCAM Visual Attributions
Generates visual explanation heatmaps using actual images from the test split.


In [ ]:
from src.explainability.layercam import LayerCAM

visual_configs = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'trustoct')


In [ ]:
if sample_path and os.path.exists(sample_path):
    img_pil = Image.open(sample_path).convert('RGB')
    tensor_input = val_transform(img_pil).unsqueeze(0).to(device)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_pil)
    axes[0].set_title('Original B-Scan', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    for idx, (config_item, path, display_name) in enumerate(visual_configs):
        ax = axes[idx + 1]
        model_inst = build_model(config_item).to(device)
        if os.path.exists(path):
            model_inst.load_state_dict(torch.load(path, map_location=device))
        model_inst.eval()
        
        target_layer = model_inst.backbone.layer4[-1]
        cam_generator = LayerCAM(model_inst, target_layer)
        cam_heatmap = cam_generator.generate(tensor_input, class_idx=3) # NORMAL class target
        cam_generator.release()
        
        cam_np = cam_heatmap.detach().cpu().numpy()
        orig_cv = cv2.imread(sample_path)
        orig_cv = cv2.resize(orig_cv, (224, 224))
        heatmap_color = cv2.applyColorMap(np.uint8(255 * cam_np), cv2.COLORMAP_JET)
        overlay_img = cv2.addWeighted(orig_cv, 0.6, heatmap_color, 0.4, 0)
        
        overlay_rgb = cv2.cvtColor(overlay_img, cv2.COLOR_BGR2RGB)
        ax.imshow(overlay_rgb)
        ax.set_title(display_name, fontsize=12, fontweight='bold')
        ax.axis('off')
        
    # Save LayerCAM visual overlay directly inside trustoct output directory
    os.makedirs('outputs/trustoct/layercam', exist_ok=True)
    plt.tight_layout()
    plt.savefig('outputs/trustoct/layercam/layercam_attribution.png', dpi=300)
    plt.show()
else:
    print('Sample image missing. Skipping LayerCAM overlays.')


<a id='section10'></a>
## SECTION 10: Faithfulness Evaluation (Deletion & Insertion)
Runs progressive deletion and insertion perturbation tests on the test dataset to compute AOPC. These quantitative audits confirm if the LayerCAM attribution matches functional diagnostics.


In [ ]:
from src.explainability.comparison import calculate_saliency_entropy, run_deletion_test, run_insertion_test

test_records = test_df.to_dict('records')[:20] # Run on first 20 for speed validation
comparison_rows = []

for config_item, path, display_name in models_to_evaluate:
    print(f"Auditing explainability for {display_name}...")
    model_inst = build_model(config_item).to(device)
    is_ev = (config_item['model']['head'] == 'evidential')
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
    model_inst.eval()
    
    target_layer = model_inst.backbone.layer4[-1]
    cam_generator = LayerCAM(model_inst, target_layer)
    
    del_scores, ins_scores, entropies = [], [], []
    for rec in test_records:
        img_p = rec['image_path']
        lbl = int(rec['label'])
        
        if not os.path.exists(img_p):
            continue
        try:
            img_p_pil = Image.open(img_p).convert('RGB')
            tensor_inp = val_transform(img_p_pil).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outs = model_inst(tensor_inp)
                if is_ev:
                    from src.models.edl_head import get_evidence_metrics
                    probs_ev, _ = get_evidence_metrics(outs)
                    pred_cls = probs_ev.argmax(dim=1).item()
                else:
                    pred_cls = torch.softmax(outs, dim=1).argmax(dim=1).item()
                    
            cam_heatmap = cam_generator.generate(tensor_inp, class_idx=pred_cls)
            _, aopc_del, _ = run_deletion_test(model_inst, tensor_inp, cam_heatmap, class_idx=pred_cls)
            _, aopc_ins, _ = run_insertion_test(model_inst, tensor_inp, cam_heatmap, class_idx=pred_cls)
            entropy = calculate_saliency_entropy(cam_heatmap)
            
            del_scores.append(aopc_del)
            ins_scores.append(aopc_ins)
            entropies.append(entropy)
        except Exception:
            continue
            
    cam_generator.release()
    comparison_rows.append({
        'Model': display_name,
        'Deletion AOPC': f"{np.mean(del_scores):.4f} ± {np.std(del_scores):.4f}",
        'Insertion AOPC': f"{np.mean(ins_scores):.4f} ± {np.std(ins_scores):.4f}",
        'Saliency Entropy': f"{np.mean(entropies):.4f} ± {np.std(entropies):.4f}"
    })
    
table_4_df = pd.DataFrame(comparison_rows)
print('\n--- TABLE 4: EXPLAINABILITY FAITHFULNESS BENCHMARKS (TEST-SET AVERAGES) ---')
display(table_4_df)
table_4_df.to_csv('results/tables/table_4_explainability_averages.csv', index=False)


<a id='section11'></a>
## SECTION 11: External Validation (OCTID)
Evaluates generalization on the out-of-distribution **OCTID cohort** using the trained AE-ResNet model.


In [ ]:
from src.evaluation.cross_dataset import run_external_validation

octid_csv = 'octid_dataset_mapping.csv'
if not os.path.exists(octid_csv):
    octid_csv = '/content/drive/MyDrive/octid_dataset_mapping.csv'
    
if os.path.exists(octid_csv):
    octid_df = pd.read_csv(octid_csv)
    sample_path_id = octid_df.iloc[0]['image_path']
    if not os.path.exists(sample_path_id):
        def correct_id_path(win_p):
            l_p = win_p.replace('\\\\', '/')
            r_p = l_p[l_p.find('OCTID/'):] if 'OCTID/' in l_p else '/'.join(l_p.split('/')[-3:])
            return os.path.join(drive_base, r_p)
        octid_df['image_path'] = octid_df['image_path'].apply(correct_id_path)
        sample_path_id = octid_df.iloc[0]['image_path']
    
    external_rows = []
    for config_item, path, display_name in models_to_evaluate:
        is_ev = (config_item['model']['head'] == 'evidential')
        model_inst = build_model(config_item).to(device)
        if os.path.exists(path):
            model_inst.load_state_dict(torch.load(path, map_location=device))
        model_inst.eval()
        
        print(f'Evaluating generalization on OCTID for {display_name}...')
        res = run_external_validation(
            model=model_inst, df_external=octid_df, batch_size=16,
            apply_bilateral=True, apply_clahe=(config_item['model']['dg']=='coral'),
            apply_min_max=(config_item['model']['dg']=='coral'), is_evidential=is_ev, device_name=str(device)
        )
        
        external_rows.append({
            'Model': display_name,
            'Accuracy': res['metrics']['accuracy'],
            'Precision': res['metrics']['precision'],
            'Recall': res['metrics']['recall'],
            'Macro F1': res['metrics']['f1_score'],
            'ECE': res['metrics']['ece'],
            'Brier': res['metrics']['brier_score']
        })
        
    table_5_df = pd.DataFrame(external_rows)
    print('\n--- TABLE 5: CROSS-SCANNER DOMAIN GENERALIZATION (OCTID) ---')
    display(table_5_df)
    table_5_df.to_csv('results/tables/table_5_external_validation.csv', index=False)
else:
    print('OCTID mapping CSV not found. Skipping external validation.')


<a id='section12'></a>
## SECTION 12: Statistical Analysis & Wilcoxon Significance
Computes Wilcoxon signed-rank tests dynamically from repeated multi-seed experimental runs.


In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

print('Running Statistical Significance Tests...')
results_cache = {}

for config_item, path, display_name in models_to_evaluate:
    model_inst = build_model(config_item).to(device)
    is_ev = (config_item['model']['head'] == 'evidential')
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
    model_inst.eval()
    
    preds, labels = [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model_inst(images)
            if is_ev:
                from src.models.edl_head import get_evidence_metrics
                probs, _ = get_evidence_metrics(outputs)
            else:
                probs = torch.softmax(outputs, dim=1)
            _, p = torch.max(probs, 1)
            preds.extend(p.cpu().numpy())
            labels.extend(targets.numpy())
            
    results_cache[display_name] = {'preds': np.array(preds), 'labels': np.array(labels)}

def run_mcnemar(labels, preds_a, preds_b):
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    res = mcnemar(table, exact=True)
    return res.statistic, res.pvalue

proposed_name = 'TrustOCT (Proposed)'
if proposed_name in results_cache and 'ResNet-50 Baseline' in results_cache:
    prop_data = results_cache[proposed_name]
    base_data = results_cache['ResNet-50 Baseline']
    stat, p_val = run_mcnemar(prop_data['labels'], prop_data['preds'], base_data['preds'])
    print(f"McNemar's test proposed vs baseline ResNet50 p-value: {p_val:.5f}")
    
    # Wilcoxon signed-rank test
    diff = (prop_data['preds'] == prop_data['labels']).astype(int) - (base_data['preds'] == base_data['labels']).astype(int)
    if not np.all(diff == 0):
        stat_w, p_val_w = wilcoxon(diff)
        print(f"Wilcoxon proposed vs baseline ResNet50 p-value: {p_val_w:.5f}")


<a id='section13'></a>
## SECTION 13: Publication Paper Assets
Creates directories and compiles unified paper figures (Learning Curves, Confusion Matrix grids, Calibration Curves) and Zips results.


In [ ]:
import shutil
from src.evaluation.plots import plot_reliability_diagram

print('Generating publication figure curves...')
history_files = {
    'resnet50': 'outputs/resnet50/metrics.csv',
    'trustoct': 'outputs/trustoct/metrics.csv'
}

# 1. Unified training curves comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    ('train_loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('train_acc', 'Training Accuracy', axes[1, 0]),
    ('val_acc', 'Validation Accuracy', axes[1, 1])
]
for metric_key, title, ax in metrics_to_plot:
    for model_name, path in history_files.items():
        if os.path.exists(path):
            df_hist = pd.read_csv(path)
            ax.plot(df_hist['epoch'], df_hist[metric_key], label=model_name, linewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend()
plt.tight_layout()
plt.savefig('results/figures/figure_2_training_curves.png', dpi=300)
plt.show()

# 2. Absolute dynamic drop table (OCTDL vs OCTID)
drop_rows = []
for config_item, path, display_name in models_to_evaluate:
    is_ev = (config_item['model']['head'] == 'evidential')
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
    model_inst.eval()
    
    # Source performance
    src_preds, src_labels = [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model_inst(images)
            if is_ev:
                probs, _ = get_evidence_metrics(outputs)
            else:
                probs = torch.softmax(outputs, dim=1)
            _, p = torch.max(probs, 1)
            src_preds.extend(p.cpu().numpy())
            src_labels.extend(targets.numpy())
    src_acc = accuracy_score(src_labels, src_preds) * 100
    
    # Target performance
    tgt_acc = 0.0
    if os.path.exists(octid_csv):
        res_id = run_external_validation(
            model=model_inst, df_external=octid_df, batch_size=16,
            apply_bilateral=True, apply_clahe=(config_item['model']['dg']=='coral'),
            apply_min_max=(config_item['model']['dg']=='coral'), is_evidential=is_ev, device_name=str(device)
        )
        tgt_acc = res_id['metrics']['accuracy'] * 100
        
    acc_drop = tgt_acc - src_acc
    drop_rows.append({
        'Model': display_name,
        'Source (OCTDL) Acc (%)': f'{src_acc:.2f}%',
        'Target (OCTID) Acc (%)': f'{tgt_acc:.2f}%',
        'Absolute Performance Drop': f'{acc_drop:.2f}%'
    })
    
drop_df = pd.DataFrame(drop_rows)
print('\n--- TABLE 5.5: CROSS-DOMAIN ACCURACY DECAY (DOMAIN SHIFT DROP) ---')
display(drop_df)
drop_df.to_csv('results/tables/table_5_5_domain_drop.csv', index=False)

# Zip results
zip_path = '/content/final_paper_results'
if os.path.exists('outputs'):
    shutil.make_archive(zip_path, 'zip', 'outputs')
    print("Success: Zip archive 'final_paper_results.zip' created successfully from outputs folder!")


In [ ]:
# Complexity Analysis
import time

complexity_rows = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)

for config_item, path, display_name in models_to_evaluate:
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
    model_inst.eval()
    
    total_params = sum(p.numel() for p in model_inst.parameters())
    trainable_params = sum(p.numel() for p in model_inst.parameters() if p.requires_grad)
    
    if os.path.exists(path):
        model_size_mb = os.path.getsize(path) / (1024 * 1024)
    else:
        model_size_mb = 0.0
        
    # Latency evaluation
    with torch.no_grad():
        for _ in range(20):
            _ = model_inst(dummy_input)
            
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_t = time.perf_counter()
        for _ in range(100):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_t = time.perf_counter()
    avg_inf_ms = ((end_t - start_t) / 100) * 1000
    
    complexity_rows.append({
        'Model': display_name,
        'Total Params (M)': f"{total_params / 1e6:.2f}M",
        'Trainable Params (M)': f"{trainable_params / 1e6:.2f}M",
        'Size on Disk (MB)': f"{model_size_mb:.2f} MB" if model_size_mb > 0 else "-",
        'Inference Speed (ms)': f"{avg_inf_ms:.2f} ms"
    })
    
complexity_df = pd.DataFrame(complexity_rows)
print('\n--- TABLE 6: COMPUTATIONAL COMPLEXITY ANALYSIS ---')
display(complexity_df)
complexity_df.to_csv('results/tables/table_6_computational_complexity.csv', index=False)


In [ ]:
# Representative Failure Analysis Grid
model_proposed = build_model(models_to_evaluate[5][0]).to(device)
if os.path.exists(models_to_evaluate[5][1]):
    model_proposed.load_state_dict(torch.load(models_to_evaluate[5][1], map_location=device))
model_proposed.eval()

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
misclass_records = []
with torch.no_grad():
    for idx, (images, targets) in enumerate(test_loader):
        images = images.to(device)
        outputs = model_proposed(images)
        # Evidential head prediction
        from src.models.edl_head import get_evidence_metrics
        probs, _ = get_evidence_metrics(outputs)
        preds = torch.argmax(probs, dim=1)
        
        preds_np = preds.cpu().numpy()
        targets_np = targets.numpy()
        
        for batch_idx in range(len(targets_np)):
            global_idx = idx * 16 + batch_idx
            if preds_np[batch_idx] != targets_np[batch_idx]:
                img_p = test_df.iloc[global_idx]['image_path']
                misclass_records.append({
                    'image_path': img_p,
                    'true_label': CLASSES[targets_np[batch_idx]].upper(),
                    'pred_label': CLASSES[preds_np[batch_idx]].upper()
                })
                
print(f'Total misclassified images found in test set: {len(misclass_records)}')

n_plots = min(16, len(misclass_records))
if n_plots > 0:
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes_flat = axes.flatten()
    for i in range(16):
        ax = axes_flat[i]
        if i < n_plots:
            rec = misclass_records[i]
            if os.path.exists(rec['image_path']):
                img_pil_mis = Image.open(rec['image_path']).convert('RGB')
                ax.imshow(img_pil_mis)
            ax.set_title(f"True: {rec['true_label']}\nPred: {rec['pred_label']}", fontsize=10, fontweight='bold', color='red')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('results/figures/figure_6_failure_analysis.png', dpi=300)
    plt.show()
    print("Figure 6 (Failure Analysis Grid) successfully generated!")
